# V46 Colab版 — refinerをfree-variant向けに再学習(fold0検証)

free variant(交差制約なし)は追加学習なしでfold0 OOF PQ **0.4361**まで到達
(基準0.4393にΔ-0.0032)。閾値調整はほぼ頭打ちなので、refiner自体を
**YOLO候補のbboxを中心にクロップしGTを予測する**よう再学習する
(現行はGTのbboxを中心にクロップしており、訓練/推論の不一致がある)。

**採用判定はしない。** fold0 OOF PQが基準0.4393を上回ったら fold1/2 展開を検討する。

ランタイム: **T4/L4 GPU** を選択すること。


In [ ]:
# [1/6] 環境確認・依存関係・Drive mount
import json, math, shutil, time
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from google.colab import drive
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from pycocotools import mask as mask_utils

drive.mount("/content/drive")

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("gpu:", gpu_name)
if not torch.cuda.is_available():
    raise RuntimeError("GPUランタイムを選択してください")

class CFG:
    V30_ROOT = Path("/content/drive/MyDrive/solar_filament_v30")
    V34_ROOT = Path("/content/drive/MyDrive/solar_filament_v34")
    V35_ROOT = Path("/content/drive/MyDrive/solar_filament_v35")
    ARTIFACTS_ROOT = V30_ROOT / "artifacts"
    PREPROC_DIR = V30_ROOT / "preprocessed_v30_q96"
    FOLDS_CSV = ARTIFACTS_ROOT / "grouped_folds.csv"
    TRAIN_JSON = (V30_ROOT / "competition_data" / "MAGFiLO_1.0_Kaggle_2026"
                  / "train" / "MAGFiLO_1.0_Annotations_kaggle2026_train.json")
    MANIFEST = V35_ROOT / "refiner_retrain" / "refiner_retrain_manifest_fold0.json"
    OOF_CACHE = V34_ROOT / "prediction_cache" / "oof"
    OLD_REFINER_CKPT = ARTIFACTS_ROOT / "mediseg_refiner_fold0_resume.pt"
    WORK = Path("/content/v46_refiner_retrain")
    OUT_CKPT = V35_ROOT / "refiner_retrain" / "mediseg_refiner_free_fold0.pt"
    FOLD = 0
    CONF = 0.26
    MIN_AREA = 20
    CROP_SIZE = 320
    REFINER_LR = 2e-4
    REFINER_EPOCHS = 14
    REFINER_BATCH_CANDIDATES = (8, 6, 4, 2)
    FREE_THRESHOLD = 0.86
    DROP_THRESHOLD = 0.90
    BASELINE_FOLD0_PQ = 0.4393
    FREE_BASELINE_FOLD0_PQ = 0.4361  # 現行refinerでのfree@0.86(基準)

for required in (CFG.ARTIFACTS_ROOT, CFG.PREPROC_DIR, CFG.FOLDS_CSV, CFG.TRAIN_JSON, CFG.MANIFEST):
    if not required.exists():
        raise FileNotFoundError(f"{required} が見つからない。Driveのマウント・パスを確認")

CFG.WORK.mkdir(parents=True, exist_ok=True)
CFG.OUT_CKPT.parent.mkdir(parents=True, exist_ok=True)
print({"images": len(list(CFG.PREPROC_DIR.glob("*.jpeg"))), "gpu": gpu_name})


In [ ]:
# [2/6] MediSeg UNetアーキテクチャ(solar_filament/models/mediseg_models.py と同一)
"""
MediSeg-AI Knowledge Port for Solar Filament Segmentation
Includes:
1. MediSeg UNet
2. MediSeg DeepLabV3Plus with ASPP (Atrous Spatial Pyramid Pooling)
3. CombinedLoss (BCE + Dice Loss)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional

# ============================================================
# 1. Combined Loss (BCE + Dice Loss)
# ============================================================
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs = probs.view(-1)
        targets = targets.view(-1)
        
        intersection = (probs * targets).sum()
        dice = (2.0 * intersection + self.smooth) / (probs.sum() + targets.sum() + self.smooth)
        return 1.0 - dice

class CombinedLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        dice_loss = self.dice(logits, targets)
        return self.bce_weight * bce_loss + self.dice_weight * dice_loss

# ============================================================
# 2. MediSeg UNet Architecture
# ============================================================
class DoubleConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, mid_channels: Optional[int] = None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.maxpool_conv(x)

class Up(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, bilinear: bool = True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        x1 = self.up(x1)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class MediSegUNet(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 1, bilinear: bool = False):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.outc(x)

# ============================================================
# 3. MediSeg DeepLabV3Plus with ASPP Module
# ============================================================
class ASPP(nn.Module):
    def __init__(self, in_channels: int, out_channels: int = 256):
        super().__init__()
        self.conv1x1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn1x1 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=6, dilation=6, bias=False)
        self.bn3x3_1 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_2 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=12, dilation=12, bias=False)
        self.bn3x3_2 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_3 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=18, dilation=18, bias=False)
        self.bn3x3_3 = nn.BatchNorm2d(out_channels)
        
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv_global = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn_global = nn.BatchNorm2d(out_channels)
        
        self.conv_final = nn.Conv2d(out_channels * 5, out_channels, kernel_size=1, bias=False)
        self.bn_final = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        size = x.shape[2:]
        x1 = self.relu(self.bn1x1(self.conv1x1(x)))
        x2 = self.relu(self.bn3x3_1(self.conv3x3_1(x)))
        x3 = self.relu(self.bn3x3_2(self.conv3x3_2(x)))
        x4 = self.relu(self.bn3x3_3(self.conv3x3_3(x)))
        
        x5 = self.global_avg_pool(x)
        x5 = self.relu(self.bn_global(self.conv_global(x5)))
        x5 = F.interpolate(x5, size=size, mode='bilinear', align_corners=True)
        
        x_cat = torch.cat([x1, x2, x3, x4, x5], dim=1)
        return self.relu(self.bn_final(self.conv_final(x_cat)))

class MediSegDeepLabV3Plus(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 1):
        super().__init__()
        # Feature Extractor
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 512, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True)
        )
        self.aspp = ASPP(512, 256)
        self.classifier = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, out_channels, kernel_size=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input_size = x.shape[2:]
        feat = self.stem(x)
        aspp_feat = self.aspp(feat)
        out = self.classifier(aspp_feat)
        return F.interpolate(out, size=input_size, mode='bilinear', align_corners=True)

def soft_dice_loss(logits, targets):
    probs = logits.sigmoid()
    intersection = (probs * targets).sum((1, 2, 3))
    denominator = probs.sum((1, 2, 3)) + targets.sum((1, 2, 3))
    return (1 - (2 * intersection + 1) / (denominator + 1)).mean()

def boundary_map(tensor):
    dilation = torch.nn.functional.max_pool2d(tensor, 5, stride=1, padding=2)
    erosion = -torch.nn.functional.max_pool2d(-tensor, 5, stride=1, padding=2)
    return torch.clamp(dilation - erosion, 0, 1)

def refiner_loss(logits, targets):
    region = 0.35 * torch.nn.functional.binary_cross_entropy_with_logits(logits, targets) \
             + 0.65 * soft_dice_loss(logits, targets)
    edge = torch.nn.functional.l1_loss(boundary_map(logits.sigmoid()), boundary_map(targets))
    return region + 0.15 * edge


In [ ]:
# [3/6] manifestからProposalCropDataset(YOLO候補のbboxを中心にGTを予測)を作る
def imread_unicode(path):
    return cv2.imdecode(np.frombuffer(Path(path).read_bytes(), np.uint8), cv2.IMREAD_COLOR)

with open(CFG.TRAIN_JSON, encoding="utf-8") as handle:
    coco = json.load(handle)
images_by_id = {i["id"]: i for i in coco["images"]}
ann_by_id = {a["id"]: a for a in coco["annotations"]}

def ann_mask(ann, height, width):
    mask = np.zeros((height, width), dtype=np.uint8)
    for polygon in ann.get("segmentation", []):
        if len(polygon) < 6:
            continue
        points = np.rint(np.asarray(polygon).reshape(-1, 2)).astype(np.int32)
        cv2.fillPoly(mask, [points], 1)
    return mask

with open(CFG.MANIFEST, encoding="utf-8") as handle:
    manifest = json.load(handle)
samples_raw = manifest["samples"]
print({"total_samples": len(samples_raw),
       "negative": sum(1 for s in samples_raw if s["is_negative"])})

# 90/10でtrain/validに分割(manifest全体をシャッフル済みなので先頭からで良い)
split_index = int(len(samples_raw) * 0.9)
train_samples = samples_raw[:split_index]
valid_samples = samples_raw[split_index:]
print({"train": len(train_samples), "valid": len(valid_samples)})

class ProposalCropDataset(Dataset):
    def __init__(self, samples, augment=False):
        self.samples = samples
        self.augment = augment

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]
        image = imread_unicode(CFG.PREPROC_DIR / sample["file_name"])
        x0, y0, x1, y1 = sample["box"]
        if sample["is_negative"]:
            target = np.zeros((y1 - y0, x1 - x0), dtype=np.uint8)
        else:
            ann = ann_by_id[sample["target_ann_id"]]
            info = images_by_id[sample["target_image_id"]]
            full_target = ann_mask(ann, int(info["height"]), int(info["width"]))
            target = full_target[y0:y1, x0:x1]

        crop = cv2.resize(image[y0:y1, x0:x1], (CFG.CROP_SIZE, CFG.CROP_SIZE),
                          interpolation=cv2.INTER_LINEAR)
        target = cv2.resize(target, (CFG.CROP_SIZE, CFG.CROP_SIZE), interpolation=cv2.INTER_NEAREST)
        crop_t = torch.from_numpy(crop.transpose(2, 0, 1).copy()).float() / 255.0
        target_t = torch.from_numpy(target[None].copy()).float()
        if self.augment:
            k = np.random.randint(4)
            crop_t, target_t = torch.rot90(crop_t, k, (1, 2)), torch.rot90(target_t, k, (1, 2))
            if np.random.random() < 0.5:
                crop_t, target_t = torch.flip(crop_t, (2,)), torch.flip(target_t, (2,))
            if np.random.random() < 0.5:
                crop_t, target_t = torch.flip(crop_t, (1,)), torch.flip(target_t, (1,))
        return crop_t, target_t

train_ds = ProposalCropDataset(train_samples, augment=True)
valid_ds = ProposalCropDataset(valid_samples, augment=False)
print({"train_ds": len(train_ds), "valid_ds": len(valid_ds)})


In [ ]:
# [4/6] refinerを学習(既存のtrain_refinerロジックと同一)
def choose_refiner_batch_size(dataset):
    for candidate in CFG.REFINER_BATCH_CANDIDATES:
        try:
            loader = DataLoader(dataset, batch_size=candidate, shuffle=True, num_workers=2, pin_memory=True)
            image, target = next(iter(loader))
            probe = MediSegUNet(in_channels=3, out_channels=1, bilinear=False).cuda()
            with torch.amp.autocast("cuda"):
                loss = refiner_loss(probe(image.cuda()), target.cuda())
            loss.backward()
            del probe, image, target, loss, loader
            torch.cuda.empty_cache()
            return candidate
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
    raise RuntimeError("No refiner batch size fits GPU memory.")

REFINER_BATCH_SIZE = choose_refiner_batch_size(train_ds)
print("REFINER_BATCH_SIZE =", REFINER_BATCH_SIZE)

def crop_dice(model, loader, threshold=0.5):
    model.eval()
    total = count = 0
    with torch.no_grad():
        for image, target in loader:
            image, target = image.cuda(non_blocking=True), target.cuda(non_blocking=True).bool()
            pred = model(image).sigmoid() >= threshold
            intersection = (pred & target).sum((1, 2, 3)).float()
            denominator = pred.sum((1, 2, 3)) + target.sum((1, 2, 3))
            total += ((2 * intersection + 1) / (denominator + 1)).sum().item()
            count += len(image)
    return total / max(count, 1)

train_started = time.perf_counter()
train_loader = DataLoader(train_ds, batch_size=REFINER_BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, persistent_workers=True)
valid_loader = DataLoader(valid_ds, batch_size=REFINER_BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True, persistent_workers=True)
model = MediSegUNet(in_channels=3, out_channels=1, bilinear=False).cuda()
# 既存(shrink向け)refinerの重みから warm start する
old_state = torch.load(CFG.OLD_REFINER_CKPT, map_location="cpu", weights_only=False)
model.load_state_dict(old_state.get("best_model") or old_state["model"], strict=True)
print("warm start: 既存refinerの重みから開始")

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.REFINER_LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.REFINER_EPOCHS)
scaler = torch.amp.GradScaler("cuda")
best_dice, best_state = -1.0, None

for epoch in range(CFG.REFINER_EPOCHS):
    model.train()
    running = 0.0
    for image, target in tqdm(train_loader, desc=f"epoch {epoch + 1}/{CFG.REFINER_EPOCHS}", leave=False):
        image, target = image.cuda(non_blocking=True), target.cuda(non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda"):
            loss = refiner_loss(model(image), target)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        scaler.step(optimizer)
        scaler.update()
        running += loss.item() * len(image)
    val_dice = crop_dice(model, valid_loader)
    if val_dice > best_dice:
        best_dice = val_dice
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    scheduler.step()
    print({"epoch": epoch + 1, "train_loss": running / len(train_ds), "crop_dice": val_dice, "best": best_dice})

model.load_state_dict(best_state)
torch.save({"model": model.state_dict(), "best_model": best_state, "best_dice": best_dice},
           CFG.OUT_CKPT)
train_seconds = time.perf_counter() - train_started
print(f"学習時間={train_seconds/60:.1f}分  best_crop_dice={best_dice:.4f}  保存先={CFG.OUT_CKPT}")


In [ ]:
# [5/6] fold0 YOLO OOFへ新refinerを適用し、free@0.86 + drop_contained@0.90でnpzキャッシュ相当を評価
def load_yolo_cache(path):
    data = np.load(path)
    _, width = map(int, data["shape"])
    packed = data["packed"]
    if packed.size == 0:
        return [], np.zeros(0, np.float32)
    return ([np.unpackbits(item, axis=-1)[:, :width].astype(bool) for item in packed],
            data["confidences"].astype(np.float32))

def filter_proposals(masks, confidences):
    keep = [i for i, (m, s) in enumerate(zip(masks, confidences))
            if s >= CFG.CONF and m.sum() >= CFG.MIN_AREA]
    return [masks[i] for i in keep]

def square_box(mask, context=1.65, minimum=96):
    height, width = mask.shape
    ys, xs = np.where(mask)
    if not len(xs):
        return 0, 0, width, height
    cx, cy = (xs.min() + xs.max() + 1) / 2, (ys.min() + ys.max() + 1) / 2
    side = int(math.ceil(max(xs.max() - xs.min() + 1, ys.max() - ys.min() + 1) * context))
    side = min(max(side, minimum), min(height, width))
    x0, y0 = int(round(cx - side / 2)), int(round(cy - side / 2))
    x1, y1 = x0 + side, y0 + side
    if x0 < 0: x1 -= x0; x0 = 0
    if y0 < 0: y1 -= y0; y0 = 0
    if x1 > width: x0 -= x1 - width; x1 = width
    if y1 > height: y0 -= y1 - height; y1 = height
    return max(0, x0), max(0, y0), min(width, x1), min(height, y1)

def refine_batch(refiner_model, image, proposals, batch=16):
    boxes, crops = [], []
    for proposal in proposals:
        x0, y0, x1, y1 = square_box(proposal)
        crop = cv2.resize(image[y0:y1, x0:x1], (CFG.CROP_SIZE, CFG.CROP_SIZE), interpolation=cv2.INTER_LINEAR)
        boxes.append((x0, y0, x1, y1))
        crops.append(crop.transpose(2, 0, 1))
    probabilities = []
    with torch.no_grad():
        for start in range(0, len(crops), batch):
            chunk = np.stack(crops[start:start + batch])
            tensor = torch.from_numpy(chunk.copy()).float().cuda() / 255.0
            output = refiner_model(tensor).sigmoid()[:, 0].cpu().numpy()
            for offset in range(output.shape[0]):
                x0, y0, x1, y1 = boxes[start + offset]
                probabilities.append(cv2.resize(output[offset], (x1 - x0, y1 - y0), interpolation=cv2.INTER_LINEAR))
    return boxes, probabilities

def free_variant(proposal, box, probability, threshold):
    x0, y0, x1, y1 = box
    refined = np.zeros_like(proposal, dtype=bool)
    refined[y0:y1, x0:x1] = probability >= threshold
    return refined

def greedy_disjoint_local(masks, order, min_area=CFG.MIN_AREA, canvas=2048):
    occupied = np.zeros((canvas, canvas), dtype=bool)
    kept = []
    for index in order:
        exclusive = masks[index] & ~occupied
        if exclusive.sum() < min_area:
            continue
        kept.append(exclusive)
        occupied |= exclusive
    return kept

def drop_contained_indices_local(masks, threshold):
    if len(masks) < 2:
        return list(range(len(masks)))
    rles = mask_utils.encode(np.asfortranarray(np.stack(masks, axis=-1).astype(np.uint8)))
    areas = np.asarray(mask_utils.area(rles), dtype=np.float64)
    self_iou = np.asarray(mask_utils.iou(rles, rles, [0] * len(rles)), dtype=np.float64)
    dropped = set()
    for i in range(len(masks)):
        for j in range(i + 1, len(masks)):
            if i in dropped or j in dropped:
                continue
            value = self_iou[i, j]
            if value <= 0:
                continue
            intersection = value * (areas[i] + areas[j]) / (1.0 + value)
            smaller = i if areas[i] <= areas[j] else j
            if intersection / max(areas[smaller], 1.0) >= threshold:
                dropped.add(smaller)
    return [i for i in range(len(masks)) if i not in dropped]

def apply_ownership_local(masks, threshold):
    if not masks:
        return []
    areas = np.array([m.sum() for m in masks], dtype=np.float64)
    subset = drop_contained_indices_local(masks, threshold)
    return greedy_disjoint_local(masks, sorted(subset, key=lambda i: areas[i]))

fold_frame = pd.read_csv(CFG.FOLDS_CSV)
fold_of_file = dict(zip(fold_frame["file_name"], fold_frame["fold"].astype(int)))
file_to_ids = defaultdict(list)
for image_id, info in images_by_id.items():
    file_to_ids[info["file_name"]].append(image_id)
anns_by_image = defaultdict(list)
for ann in coco["annotations"]:
    anns_by_image[ann["image_id"]].append(ann)

fold0_files = sorted(
    n for n in file_to_ids
    if fold_of_file.get(n) == CFG.FOLD and (CFG.PREPROC_DIR / n).exists()
    and (CFG.OOF_CACHE / f"yolo_fold_{CFG.FOLD}" / f"{Path(n).stem}.npz").exists()
)
print(f"fold0観測数: {len(fold0_files)}")

model.eval()
cache_dir = CFG.WORK / "oof_cache_newrefiner"
started = time.perf_counter()
for position, name in enumerate(fold0_files, 1):
    out_path = cache_dir / f"{Path(name).stem}.npz"
    if out_path.exists():
        continue
    image = imread_unicode(CFG.PREPROC_DIR / name)
    masks, confidences = load_yolo_cache(CFG.OOF_CACHE / f"yolo_fold_{CFG.FOLD}" / f"{Path(name).stem}.npz")
    proposals = filter_proposals(masks, confidences)
    if not proposals:
        out_path.parent.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(out_path, packed=np.empty((0, 0, 0), np.uint8),
                            shape=np.asarray([2048, 2048], np.int32), confidences=np.zeros(0, np.float32))
        continue
    boxes, probabilities = refine_batch(model, image, proposals)
    refined = [free_variant(proposals[i], boxes[i], probabilities[i], CFG.FREE_THRESHOLD)
               for i in range(len(proposals))]
    final = apply_ownership_local(refined, CFG.DROP_THRESHOLD)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if final:
        packed = np.packbits(np.stack(final).astype(np.uint8), axis=-1)
        shape = np.asarray(final[0].shape, np.int32)
    else:
        packed = np.empty((0, 0, 0), np.uint8)
        shape = np.asarray([2048, 2048], np.int32)
    np.savez_compressed(out_path, packed=packed, shape=shape,
                        confidences=np.ones(len(final), np.float32))
    if position % 25 == 0 or position == len(fold0_files):
        print(f"  {position}/{len(fold0_files)}  {time.perf_counter() - started:.0f}s", flush=True)
print(cache_dir)


In [ ]:
# [6/6] 公式PQで新refiner+free@0.86の結果を採点し、基準0.4393・現行free@0.86の0.4361と比較
"""公式評価指標のローカル実装 (Solar Filament Segmentation Challenge 2026).

## リーダーボード指標 = Panoptic Quality (PQ) 【確認済み】

2026-08-07 に評価方法が変更され、2026-08-12 にリーダーボードが再採点された。
それ以前は Dice (torchmetrics.segmentation.DiceScore) だった。現行は PQ。

    PQ = sum(IoU of TP pairs) / (|TP| + 0.5*|FP| + 0.5*|FN|),  TP は IoU > 0.5

出典は主催者が公開した自己評価ノートブック (Apache-2.0):
https://www.kaggle.com/code/azimahmadzadeh/self-evaluation-notebook
本モジュールの PQ 実装は、その `get_pq_score` / `fp_count_hit` / `fn_count_hit`
の挙動をそのまま再現する。原典は Kirillov et al., CVPR 2019 (10.1109/CVPR.2019.00963)
だが、**公式実装は原典そのままではない**ので、原典ではなく公式実装に合わせること。

公式実装で特に効く仕様:

  1. **IoU > 0.5 は硬い崖。** IoU 0.5 は Dice 0.667 に相当する。これを下回る予測は
     得点に一切寄与せず、FP として分母を 0.5 増やし、対応する GT も FN として
     さらに 0.5 増やす。惜しい予測は「無得点」ではなく「二重の減点」になる。
  2. **評価単位は (annotator, image) ペア。** GT の filament_id は
     `<annotator_id>-<image_id>_<n>` 形式で、`annotator_image` ごとに 1 行を作り、
     同一の予測集合を各アノテータの GT と個別に突き合わせて、TP/FP/FN を
     全体へ足し込む。複数アノテータ画像は事実上その人数ぶん重み付けされる。
     variant ごとにスコアを出して平均するのではない (V30/V31 の扱いとは別物)。
  3. **分母は全画像を通した micro 集計。** 画像単位のマクロ平均ではないので、
     個体数の多い画像が支配的になる。
  4. **FP は「どの GT とも IoU > 0.5 にならない予測」。** アノテータごとに数える。

## 副次指標

MIoU (Multiscale IoU) も実装してあるが、**リーダーボード指標ではない**。
IEEE Big Data Cup の紹介ページには MIoU が挙がっているものの、Kaggle の
Evaluation は PQ と Dice/IoU 分布・一対多/多対一分布で構成される。
MIoU は細構造 (barb) の再現性を測る補助指標、および技術報告書 (評価の 30% を
占める定性評価) の裏付けとして使う位置づけに留める。

  - MIoU: Ahmadzadeh, Kempton, Chen, Angryk, ICIP 2021 (arXiv:2105.14572)
    参照実装 https://bitbucket.org/gsudmlab/multiscale_iou/
  - EdgeAttNet (arXiv:2509.02964) は MAGFiLO test split で
    mIoU_pairwise 0.6451 / mIoU_multiscale 0.7032 を報告

## 過去数値との接続

V19〜V34 が使っていた Hungarian matched Dice は PQ ではない。normalization も
(max(|GT|,|pred|) vs TP+0.5FP+0.5FN)、通貨も (Dice vs IoU)、閾値の有無も違う。
過去の OOF 数値と本モジュールの PQ を直接比較しないこと。接続用に
hungarian_dice() を残してあるので、必ず指標名を明示して併記する。
"""


import numpy as np
from scipy.optimize import linear_sum_assignment

# Δ = {2^n : n < 10} = {1, 2, 4, ..., 512} (原論文 4章の実験設定)
DEFAULT_DELTAS: tuple[int, ...] = tuple(2 ** n for n in range(10))

# 原論文は contour を推奨。EdgeAttNet は mask と記述。既定は原論文に合わせる。
USE_CONTOUR_DEFAULT = True

METRIC_VERSION = "official-v1-2026-08-15"


def _as_bool(mask: np.ndarray) -> np.ndarray:
    return np.asarray(mask).astype(bool)


def contour_of(mask: np.ndarray) -> np.ndarray:
    """8近傍 erosion との差分で 1px の輪郭を取り出す。

    cv2/scipy に依存せず numpy のシフトだけで計算する。空 mask は空 contour を返す。
    """
    mask = _as_bool(mask)
    if not mask.any():
        return mask
    eroded = mask.copy()
    for dy in (-1, 0, 1):
        for dx in (-1, 0, 1):
            if dy == 0 and dx == 0:
                continue
            shifted = np.zeros_like(mask)
            ys = slice(max(dy, 0), mask.shape[0] + min(dy, 0))
            yd = slice(max(-dy, 0), mask.shape[0] + min(-dy, 0))
            xs = slice(max(dx, 0), mask.shape[1] + min(dx, 0))
            xd = slice(max(-dx, 0), mask.shape[1] + min(-dx, 0))
            shifted[yd, xd] = mask[ys, xs]
            eroded &= shifted
    return mask & ~eroded


def cell_reduce(mask: np.ndarray, delta: int) -> np.ndarray:
    """s(o, delta): delta x delta のセルに 1 つでも前景があれば 1 とする粗視化。"""
    mask = _as_bool(mask)
    if delta <= 1:
        return mask
    height, width = mask.shape
    padded_h = -(-height // delta) * delta
    padded_w = -(-width // delta) * delta
    if padded_h != height or padded_w != width:
        padded = np.zeros((padded_h, padded_w), dtype=bool)
        padded[:height, :width] = mask
        mask = padded
    return mask.reshape(padded_h // delta, delta, padded_w // delta, delta).any(axis=(1, 3))


def cell_keys(
    mask: np.ndarray,
    deltas: tuple[int, ...] = DEFAULT_DELTAS,
    use_contour: bool = USE_CONTOUR_DEFAULT,
) -> list[np.ndarray]:
    """mask を「各 delta で占有するセルの一意キー配列」へ前計算する。

    s(o, delta) を密な 2 次元配列ではなく座標集合として持つ。輪郭画素は画像
    全体に比べて極端に疎なので、計算量が画像面積 O(H*W) から輪郭長 O(P) に落ちる。
    2048x2048 の総当たり評価が現実的な時間で終わるのはこの前計算のおかげであり、
    V34 の OOF 評価が 524/707 で停止した経路を避けるための中核。

    グリッドは画像原点にそろえたままなので、密な cell_reduce と厳密に一致する。
    """
    mask = _as_bool(mask)
    if use_contour:
        mask = contour_of(mask)
    height, width = mask.shape
    ys, xs = np.nonzero(mask)
    keys = []
    for delta in deltas:
        if ys.size == 0:
            keys.append(np.empty(0, dtype=np.int64))
            continue
        cells_per_row = -(-width // delta)
        flat = (ys // delta).astype(np.int64) * cells_per_row + (xs // delta)
        keys.append(np.unique(flat))
    return keys


def multiscale_iou_from_keys(
    truth_keys: list[np.ndarray],
    prediction_keys: list[np.ndarray],
) -> float:
    """cell_keys() の出力どうしで MIoU を計算する。"""
    ratios = []
    for truth_cells, prediction_cells in zip(truth_keys, prediction_keys):
        if truth_cells.size == 0:
            ratios.append(0.0)
            continue
        shared = np.intersect1d(truth_cells, prediction_cells, assume_unique=True).size
        ratios.append(float(shared) / truth_cells.size)
    if not ratios:
        return 0.0
    return float(np.mean(ratios))


def multiscale_iou(
    ground_truth: np.ndarray,
    prediction: np.ndarray,
    deltas: tuple[int, ...] = DEFAULT_DELTAS,
    use_contour: bool = USE_CONTOUR_DEFAULT,
) -> float:
    """MIoU(o, o~) = mean_delta r(o, o~, delta).

    r(o, o~, delta) = n(s(o, delta) & s(o~, delta)) / n(s(o, delta))

    非対称であることに注意: 分母は ground truth 側だけで決まる。予測が過剰に
    大きいことは r では直接罰せられない。precision 側は別指標で見ること。

    多数のペアを評価するときは cell_keys() で前計算して
    multiscale_iou_from_keys() を使うこと。こちらは輪郭を毎回作り直す。
    """
    truth = _as_bool(ground_truth)
    if use_contour:
        if not contour_of(truth).any():
            return 0.0
    elif not truth.any():
        return 0.0
    return multiscale_iou_from_keys(
        cell_keys(ground_truth, deltas, use_contour),
        cell_keys(prediction, deltas, use_contour),
    )


def iou(first: np.ndarray, second: np.ndarray) -> float:
    first = _as_bool(first)
    second = _as_bool(second)
    union = int(np.logical_or(first, second).sum())
    if union == 0:
        return 0.0
    return float(np.logical_and(first, second).sum()) / union


def dice(first: np.ndarray, second: np.ndarray) -> float:
    first = _as_bool(first)
    second = _as_bool(second)
    denominator = int(first.sum()) + int(second.sum())
    if denominator == 0:
        return 0.0
    return 2.0 * float(np.logical_and(first, second).sum()) / denominator


def pairwise_iou(
    gt_masks: list[np.ndarray],
    pred_masks: list[np.ndarray],
    zero_intersection: str = "exclude",
) -> float:
    """交差を持つ (gt, pred) ペアの IoU 平均 (EdgeAttNet の IoU_pairwise)。

    zero_intersection="exclude" は本文記述 (交差のあるペアのみ評価) に、
    "zero" は式の otherwise=0 に対応する。検出漏れ・過剰検出は
    この指標では罰せられないので、必ず hit/miss rate と併記すること。
    """
    if not gt_masks or not pred_masks:
        return 0.0
    scores = []
    for truth in gt_masks:
        truth = _as_bool(truth)
        for prediction in pred_masks:
            prediction = _as_bool(prediction)
            if np.logical_and(truth, prediction).any():
                scores.append(iou(truth, prediction))
            elif zero_intersection == "zero":
                scores.append(0.0)
    if not scores:
        return 0.0
    return float(np.mean(scores))


def matched_multiscale_iou(
    gt_masks: list[np.ndarray],
    pred_masks: list[np.ndarray],
    deltas: tuple[int, ...] = DEFAULT_DELTAS,
    use_contour: bool = USE_CONTOUR_DEFAULT,
    unmatched_penalty: bool = True,
) -> float:
    """個体を Hungarian で対応付けてから MIoU を平均する。

    unmatched_penalty=True では max(len(gt), len(pred)) で割るため、検出漏れと
    過剰検出の両方が罰せられる。公式が対応付けをどう行うかは未確認なので、
    これは「MIoU を instance 指標へ持ち上げた自前の変種」であることを忘れないこと。
    """
    if not gt_masks and not pred_masks:
        return 1.0
    if not gt_masks or not pred_masks:
        return 0.0
    truth_keys = [cell_keys(m, deltas, use_contour) for m in gt_masks]
    prediction_keys = [cell_keys(m, deltas, use_contour) for m in pred_masks]
    matrix = np.zeros((len(gt_masks), len(pred_masks)), dtype=np.float32)
    for i, truth in enumerate(truth_keys):
        for j, prediction in enumerate(prediction_keys):
            matrix[i, j] = multiscale_iou_from_keys(truth, prediction)
    rows, columns = linear_sum_assignment(-matrix)
    total = float(matrix[rows, columns].sum())
    divisor = max(len(gt_masks), len(pred_masks)) if unmatched_penalty else len(rows)
    return total / max(divisor, 1)


def hit_miss_rate(
    gt_masks: list[np.ndarray],
    pred_masks: list[np.ndarray],
    threshold: float = 0.5,
) -> tuple[float, float, float]:
    """(hit_rate, miss_rate, false_positive_rate) を IoU 閾値ベースで返す。

    hit  : 閾値以上の予測を 1 つ以上持つ GT の割合 (recall)
    miss : 1 - hit
    fp   : どの GT とも閾値以上で対応しない予測の割合
    """
    if not gt_masks:
        return 0.0, 0.0, 1.0 if pred_masks else 0.0
    if not pred_masks:
        return 0.0, 1.0, 0.0
    matrix = np.zeros((len(gt_masks), len(pred_masks)), dtype=np.float32)
    for i, truth in enumerate(gt_masks):
        for j, prediction in enumerate(pred_masks):
            matrix[i, j] = iou(truth, prediction)
    hits = float((matrix.max(axis=1) >= threshold).mean())
    false_positives = float((matrix.max(axis=0) < threshold).mean())
    return hits, 1.0 - hits, false_positives


def hungarian_dice(gt_masks: list[np.ndarray], pred_masks: list[np.ndarray]) -> float:
    """V31/V34 と同一の strict instance Dice。過去数値との接続用にのみ使う。"""
    if not gt_masks and not pred_masks:
        return 1.0
    if not gt_masks or not pred_masks:
        return 0.0
    matrix = np.zeros((len(gt_masks), len(pred_masks)), dtype=np.float32)
    for i, truth in enumerate(gt_masks):
        for j, prediction in enumerate(pred_masks):
            matrix[i, j] = dice(truth, prediction)
    rows, columns = linear_sum_assignment(-matrix)
    return float(matrix[rows, columns].sum() / max(len(gt_masks), len(pred_masks)))


def fragmentation_counts(
    gt_masks: list[np.ndarray],
    pred_masks: list[np.ndarray],
    overlap_fraction: float = 0.10,
) -> tuple[int, int]:
    """(splits, merges)。V34 の実装と同一定義。"""
    splits = 0
    for truth in gt_masks:
        truth = _as_bool(truth)
        area = max(int(truth.sum()), 1)
        matches = sum(
            int(np.logical_and(truth, _as_bool(p)).sum()) / area >= overlap_fraction
            for p in pred_masks
        )
        splits += max(matches - 1, 0)
    merges = 0
    for prediction in pred_masks:
        prediction = _as_bool(prediction)
        area = max(int(prediction.sum()), 1)
        matches = sum(
            int(np.logical_and(_as_bool(t), prediction).sum()) / area >= overlap_fraction
            for t in gt_masks
        )
        merges += max(matches - 1, 0)
    return splits, merges


# --------------------------------------------------------------------------
# Panoptic Quality — リーダーボード指標
# --------------------------------------------------------------------------

PQ_IOU_THRESHOLD = 0.5


def iou_matrix_of(gt_masks: list[np.ndarray], pred_masks: list[np.ndarray]) -> np.ndarray:
    """公式 get_overlap_matrices と同じ (n_gt, n_pred) の IoU 行列。"""
    if not gt_masks or not pred_masks:
        return np.zeros((len(gt_masks), len(pred_masks)), dtype=np.float64)
    gt_flat = np.stack([_as_bool(m).ravel() for m in gt_masks]).astype(np.float64)
    pred_flat = np.stack([_as_bool(m).ravel() for m in pred_masks]).astype(np.float64)
    intersection = gt_flat @ pred_flat.T
    union = gt_flat.sum(1)[:, None] + pred_flat.sum(1)[None, :] - intersection
    return np.divide(intersection, union, out=np.zeros_like(intersection), where=union > 0)


class PanopticQuality:
    """公式 get_pq_score をそのまま再現する累積器。

    公式実装は (annotator, image) ごとに 1 回 add() され、TP/FP/FN が全体へ
    足し込まれる。画像単位のマクロ平均ではないので、必ず全 annotator-image を
    同じ累積器へ入れてから score を読むこと。
    """

    def __init__(self, iou_threshold: float = PQ_IOU_THRESHOLD):
        self.iou_threshold = iou_threshold
        self.tp_ious: list[float] = []
        self.fp_count = 0
        self.fn_count = 0

    def add(self, gt_masks: list[np.ndarray], pred_masks: list[np.ndarray]) -> dict:
        """1 つの annotator-image を加算し、その行の内訳を返す。"""
        n_gt, n_pred = len(gt_masks), len(pred_masks)
        if n_gt == 0:
            self.fp_count += n_pred
            return {"n_gt": 0, "n_pred": n_pred, "tp": 0, "fp": n_pred, "fn": 0, "tp_iou_sum": 0.0}
        if n_pred == 0:
            self.fn_count += n_gt
            return {"n_gt": n_gt, "n_pred": 0, "tp": 0, "fp": 0, "fn": n_gt, "tp_iou_sum": 0.0}
        return self.add_matrix(iou_matrix_of(gt_masks, pred_masks), n_gt, n_pred)

    def add_matrix(self, matrix: np.ndarray, n_gt: int, n_pred: int) -> dict:
        """IoU 行列を計算済みの場合の加算経路。"""
        if n_gt == 0:
            self.fp_count += n_pred
            return {"n_gt": 0, "n_pred": n_pred, "tp": 0, "fp": n_pred, "fn": 0, "tp_iou_sum": 0.0}
        if n_pred == 0:
            self.fn_count += n_gt
            return {"n_gt": n_gt, "n_pred": 0, "tp": 0, "fp": 0, "fn": n_gt, "tp_iou_sum": 0.0}
        hits = matrix > self.iou_threshold
        scores = matrix[hits].tolist()
        false_positives = int((hits.sum(axis=0) == 0).sum())
        false_negatives = int((hits.sum(axis=1) == 0).sum())
        self.tp_ious.extend(scores)
        self.fp_count += false_positives
        self.fn_count += false_negatives
        return {
            "n_gt": n_gt,
            "n_pred": n_pred,
            "tp": len(scores),
            "fp": false_positives,
            "fn": false_negatives,
            "tp_iou_sum": float(sum(scores)),
        }

    @property
    def denominator(self) -> float:
        return len(self.tp_ious) + 0.5 * self.fp_count + 0.5 * self.fn_count

    @property
    def score(self) -> float:
        denominator = self.denominator
        return float(sum(self.tp_ious) / denominator) if denominator > 0 else 0.0

    def summary(self) -> dict:
        """PQ と、その内訳 (SQ x RQ)。どこが効いているかを分離して見るため。

        SQ = TP の平均 IoU (形の質)、RQ = TP/(TP+0.5FP+0.5FN) (検出の質)。
        PQ = SQ * RQ。0.35 が「形が悪い」のか「取りこぼしている」のかを区別する。
        """
        tp = len(self.tp_ious)
        segmentation_quality = float(np.mean(self.tp_ious)) if tp else 0.0
        denominator = self.denominator
        recognition_quality = float(tp / denominator) if denominator > 0 else 0.0
        return {
            "pq": self.score,
            "sq": segmentation_quality,
            "rq": recognition_quality,
            "tp": tp,
            "fp": self.fp_count,
            "fn": self.fn_count,
            "denominator": denominator,
            "iou_threshold": self.iou_threshold,
            "metric_version": METRIC_VERSION,
        }


def marginal_prediction_threshold(current_pq: float, expected_tp_iou: float = 0.7) -> float:
    """候補を 1 つ追加してよい最小の TP 確率。

    PQ = N/D のとき、TP 確率 p・想定 IoU u の候補を足すと期待 PQ は
    (N + p*u)/(D + 0.5)。これが N/D を上回る条件は p*u > 0.5*PQ。
    つまり **p > 0.5*PQ/u**。PQ=0.35, u=0.7 なら p > 0.25 で、
    「4 回に 1 回しか当たらない候補でも出した方が得」になる。

    Hungarian Dice を基準に conf を絞ってきた運用とは最適点が大きく違うので、
    conf 閾値は必ずこの式を目安に PQ 上で測り直すこと。
    """
    if expected_tp_iou <= 0:
        return 1.0
    return float(min(1.0, 0.5 * current_pq / expected_tp_iou))


def evaluate_image(
    gt_masks: list[np.ndarray],
    pred_masks: list[np.ndarray],
    deltas: tuple[int, ...] = DEFAULT_DELTAS,
    use_contour: bool = USE_CONTOUR_DEFAULT,
    iou_threshold: float = PQ_IOU_THRESHOLD,
    with_miou: bool = False,
) -> dict:
    """1 annotator-image 分の診断値。PQ 集計は別途 PanopticQuality で行うこと。

    with_miou=False が既定。MIoU はリーダーボード指標ではなく、全 OOF で回すと
    高価なので、必要なときだけ有効にする。
    """
    hit, miss, false_positive = hit_miss_rate(gt_masks, pred_masks, iou_threshold)
    splits, merges = fragmentation_counts(gt_masks, pred_masks)
    matrix = iou_matrix_of(gt_masks, pred_masks)
    near_miss = int(((matrix > 0.3) & (matrix <= iou_threshold)).any(axis=0).sum()) if matrix.size else 0
    record = {
        "metric_version": METRIC_VERSION,
        "gt_count": len(gt_masks),
        "pred_count": len(pred_masks),
        "count_error": abs(len(gt_masks) - len(pred_masks)),
        "best_iou_mean": float(matrix.max(axis=1).mean()) if matrix.size else 0.0,
        # IoU 0.3〜0.5 の「あと少しで TP だった」予測。ここが厚いなら
        # 検出は当たっていて境界品質が崖を越えられていない、という診断になる。
        "near_miss_predictions": near_miss,
        "pairwise_iou": pairwise_iou(gt_masks, pred_masks),
        "hungarian_dice": hungarian_dice(gt_masks, pred_masks),
        "hit_rate": hit,
        "miss_rate": miss,
        "false_positive_rate": false_positive,
        "splits": splits,
        "merges": merges,
    }
    if with_miou:
        record["matched_miou"] = matched_multiscale_iou(gt_masks, pred_masks, deltas, use_contour)
    return record


def to_rle(annotation, height, width):
    segmentation = annotation["segmentation"]
    if isinstance(segmentation, list):
        return mask_utils.merge(mask_utils.frPyObjects(segmentation, height, width))
    if isinstance(segmentation.get("counts"), list):
        return mask_utils.frPyObjects(segmentation, height, width)
    rle = dict(segmentation)
    if isinstance(rle["counts"], str):
        rle["counts"] = rle["counts"].encode("utf-8")
    return rle

def masks_to_rles(masks):
    if not masks:
        return []
    return mask_utils.encode(np.asfortranarray(np.stack(masks, axis=-1).astype(np.uint8)))

def iou_matrix(gt_rles, pred_rles):
    if not gt_rles or not pred_rles:
        return np.zeros((len(gt_rles), len(pred_rles)), dtype=np.float64)
    return np.asarray(mask_utils.iou(pred_rles, gt_rles, [0] * len(gt_rles)), dtype=np.float64).T

def load_cache(path):
    data = np.load(path)
    _, width = map(int, data["shape"])
    packed = data["packed"]
    if packed.size == 0:
        return [], np.zeros(0, np.float32)
    return ([np.unpackbits(item, axis=-1)[:, :width].astype(bool) for item in packed],
            data["confidences"].astype(np.float32))

def polygon_ok(ann):
    segs = [p for p in ann.get("segmentation", []) if isinstance(p, list) and len(p) >= 6]
    return bool(segs) and not ann.get("iscrowd", 0)

meter = PanopticQuality()
for name in fold0_files:
    gt_by_variant = []
    for image_id in file_to_ids[name]:
        info = images_by_id[image_id]
        gt_by_variant.append([to_rle(a, info["height"], info["width"])
                               for a in anns_by_image[image_id] if polygon_ok(a)])
    cache_path = cache_dir / f"{Path(name).stem}.npz"
    if not cache_path.exists():
        continue
    masks, confidences = load_cache(cache_path)
    # このキャッシュは既にfree+drop_contained適用済みなのでそのまま使う
    pred_rles = masks_to_rles(masks)
    for gt_rles in gt_by_variant:
        matrix = iou_matrix(gt_rles, pred_rles)
        meter.add_matrix(matrix, len(gt_rles), len(pred_rles))

summary = meter.summary()
report = {
    "scope": f"{len(fold0_files)}観測、fold0のみ。free-variant向け再学習refiner + free@{CFG.FREE_THRESHOLD} + drop_contained@{CFG.DROP_THRESHOLD}",
    **summary,
    "baseline_shrink045_fold0_pq": CFG.BASELINE_FOLD0_PQ,
    "delta_vs_baseline": round(summary["pq"] - CFG.BASELINE_FOLD0_PQ, 6),
    "beats_baseline": bool(summary["pq"] > CFG.BASELINE_FOLD0_PQ),
    "free_baseline_fold0_pq": CFG.FREE_BASELINE_FOLD0_PQ,
    "delta_vs_free_baseline": round(summary["pq"] - CFG.FREE_BASELINE_FOLD0_PQ, 6),
    "gpu_name": gpu_name,
}
local_report = CFG.WORK / "v46_refiner_retrain_fold0_report.json"
local_report.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
drive_report = CFG.V35_ROOT / "refiner_retrain" / "v46_refiner_retrain_fold0_report.json"
drive_report.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(report, ensure_ascii=False, indent=2))
print("saved:", local_report, drive_report)
